# 06-3. TCP 서버 예제

## Goal

- 요청 한 건을 처리하는 서버 경계를 분리합니다.
- 서버와 클라이언트의 자원 정리를 확인합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

`socketpair()`로 현재 프로세스 안에서만 통신하며 외부 포트를 열지 않습니다.


## Steps

### 한 요청 처리 함수 실행

서버 역할은 바이트 수 제한을 적용한 뒤 응답을 전송하고 소켓을 닫습니다.


In [1]:
import socket
import threading


def serve_once(server_socket: socket.socket, max_bytes: int = 64) -> None:
    try:
        request = server_socket.recv(max_bytes + 1)
        if len(request) > max_bytes:
            server_socket.sendall(b"ERROR:too-large")
            return
        server_socket.sendall(b"ACK:" + request.upper())
    finally:
        server_socket.close()


server_side, client_side = socket.socketpair()
worker = threading.Thread(target=serve_once, args=(server_side,), daemon=True)
worker.start()
with client_side:
    client_side.sendall(b"hello")
    response = client_side.recv(128)
worker.join(timeout=1)
print(response.decode("ascii"))


ACK:HELLO


## Checks

응답과 스레드 종료 상태를 검증합니다.


In [2]:
assert response == b"ACK:HELLO"
assert not worker.is_alive()
assert response.startswith(b"ACK:")
print("서버 경계 검사 통과")


서버 경계 검사 통과


## Next Steps

실제 `bind()`·`listen()`·`accept()` 흐름은 `examples/06-network-echo/echo_server.py`에서 확인합니다.
